## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
import numpy as np
import pandas as pd

from haystack import Pipeline, component
from hayhooks import BasePipelineWrapper
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever

from openai import OpenAI
from openai.types.create_embedding_response import CreateEmbeddingResponse

In [ ]:
from sem_rag.config.settings import VLLM, SEMDataset, Qdrant, Evaluation
from sem_rag.storage.qdrant import get_document_store
from sem_rag.retrieval.gabor import GaborRetriever
from sem_rag.embedding.embedder import CLIPEmbedder
from sem_rag.image.encoding import encode_image
from sem_rag.image.descriptions import get_crop_path

In [ ]:
INSTRUCTION = "Represent the user's input."

TOP_K = 6 # K+1 as the first will be the image itself

In [ ]:
# Lista di ID delle immagini
QUERIES = [
    "38",
]

In [ ]:
client = OpenAI(base_url=VLLM.API_URL, api_key=VLLM.API_KEY)

In [ ]:
qwen_qdrant = get_document_store(Qdrant.COLLECTION_QWEN, Qdrant.EMBEDDING_DIM_QWEN)
clip_qdrant = get_document_store(Qdrant.COLLECTION_CLIP, Qdrant.EMBEDDING_DIM_CLIP)

# Gabor

In [ ]:
gabor_retriever = GaborRetriever(Evaluation.GABOR_FILE)

In [ ]:
query_image = np.load(get_crop_path(QUERIES[0]))

In [ ]:
gabor_results = gabor_retriever.retrieve(
    query_image,
    top_k=TOP_K,
)

In [ ]:
for result in gabor_results:
    print(result)

# CLIP

In [ ]:
clip_embedder = CLIPEmbedder()
clip_retriever = QdrantEmbeddingRetriever(clip_qdrant)

In [ ]:
# Generate the CLIP text embedding
query_embedding = clip_embedder.embed_image(get_crop_path(QUERIES[0]))

In [ ]:
# Retrieve the most similar images
clip_results = clip_retriever.run(
    query_embedding=query_embedding,
    top_k=TOP_K,
).get("documents", [])

In [ ]:
for result in clip_results:
    print(result)

# RAG System

In [ ]:
@component
class QueryEmbedder():
    def __init__(self, model: str, api_base_url: str, api_key: str):
        self.model = model
        self.client = OpenAI(base_url=api_base_url, api_key=api_key)

    @component.output_types(embedding=list[float])
    def run(self, text: str = "", image_data_urls: list[str] | None = None):
        if image_data_urls:
            messages = [
                {"role": "system", "content": [{"type": "text", "text": INSTRUCTION,}]},
                {"role": "user", "content": [
                        {"type": "image_url", "image_url": {"url": image_data_urls[0]}},
                        {"type": "text", "text": ""},
                ]},
                {"role": "assistant", "content": [{"type": "text", "text": ""}]},
            ]
        else:
            messages = [
                {"role": "system", "content": [{"type": "text", "text": INSTRUCTION,}]},
                {"role": "user", "content": [{"type": "text", "text": text}]},
                {"role": "assistant", "content": [{"type": "text", "text": ""}]},
            ]

        response: CreateEmbeddingResponse = self.client.post(
            "/embeddings",
            cast_to=CreateEmbeddingResponse,
            body={
                "messages": messages,
                "model": self.model,
                "encoding_format": "float",
                "continue_final_message": True,
                "add_special_tokens": True,
            },
        )

        return {"embedding": response.data[0].embedding}

In [ ]:
class PipelineWrapper(BasePipelineWrapper):
    def setup(self) -> None:
        document_store = get_document_store(Qdrant.COLLECTION_QWEN, Qdrant.EMBEDDING_DIM_QWEN)

        self.retrieval_pipeline = Pipeline()

        self.retrieval_pipeline.add_component("query_embedder", QueryEmbedder(model=VLLM.EMBEDDING_MODEL, api_base_url=VLLM.API_URL, api_key=VLLM.API_KEY))
        self.retrieval_pipeline.add_component("retriever", QdrantEmbeddingRetriever(document_store=document_store, top_k=TOP_K))

        self.retrieval_pipeline.connect("query_embedder.embedding", "retriever")

    def retrieve_documents(self, question: str, images: list[str]):
        retrieval_run_args = {
            "query_embedder": {
                "text": question,
                "image_data_urls": images,
            },
        }

        response = self.retrieval_pipeline.run(retrieval_run_args)

        documents = (response.get("retriever", {}).get("documents", []))

        return documents

In [ ]:
qwen_retriever = PipelineWrapper()
qwen_retriever.setup()

In [ ]:
qwen_results = qwen_retriever.retrieve_documents(
    "",
    [encode_image(get_crop_path(query)) for query in QUERIES],
)

for result in qwen_results:
    print(result.id, result.content)

# Results

In [ ]:
categories_df = pd.read_csv(SEMDataset.CATEGORIES)

categories = dict(
    zip(
        categories_df["TIF_SEMImageID"].astype(str),
        categories_df["mapped_category"],
    )
)

In [ ]:
def evaluate_retrieval(query, gabor_results, clip_results, qwen_results, categories):
    query_category = categories[str(query)]

    rankings = {
        "Gabor": gabor_results[1:],
        "CLIP": [result.id for result in clip_results][1:],
        "Qwen": [result.id for result in qwen_results][1:],
    }

    rows = []

    for system, ranking in rankings.items():
        relevant = [
            categories[str(image_id)] == query_category
            for image_id in ranking
        ]

        precision = sum(relevant) / len(relevant)

        reciprocal_rank = 0.0

        for rank, is_relevant in enumerate(relevant, start=1):
            if is_relevant:
                reciprocal_rank = 1 / rank
                break

        rows.append(
            {
                "query": query,
                "category": query_category,
                "system": system,
                f"precision@{len(ranking)}": precision,
                "reciprocal_rank": reciprocal_rank,
            }
        )

    return pd.DataFrame(rows)

In [ ]:
evaluation_df = evaluate_retrieval(
    QUERIES[0],
    gabor_results,
    clip_results,
    qwen_results,
    categories=categories,
)

print(evaluation_df)

```bash
query    category system  precision@10  reciprocal_rank
   38  Biological  Gabor           0.2              1.0
   38  Biological   CLIP           0.1              1.0
   38  Biological   Qwen           0.6              1.0
```